# Filter geotweets by GV incident dates + 4-mile radius (configurable)

Loads gun-violence incidents from `gun_violence_incidents.csv` (date + address), geocodes each address to lat/lon, and keeps CGA geotweets that are:

1. **Temporal** — on incident day **t**, or **t−1** / **t−2**
2. **Spatial** — within **RADIUS_MILES** (default 4) of that incident's geocoded location

Rows with a missing `Address` are skipped. Incident years are capped at **2023**.

Paths match `chicago.ipynb` / `data_processing_pipeline.ipynb` (Harvard FAS).

Added column:
- `tag` — `t` / `t-1` / `t-2`


## 0. Setup & paths


In [ ]:
from __future__ import annotations

import glob
import gzip
import json
import os
import re
import time
from multiprocessing import Pool

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# Paths (same as chicago.ipynb / data_processing_pipeline.ipynb)
RAW_DATA_DIR = "/n/holylabs/LABS/cga/Lab/data/geo-tweets/cga-sbg-tweets/"
OUTPUT_DIR = "/n/home10/jrochatrindade/gun-violence-twitter-forecasting/data/processed/"

# Upload gun_violence_incidents.csv + optional address_to_latlon_cache.csv on the cluster
# (falls back to local cwd copies if missing)
LATLONG_PATH = "/n/home10/jrochatrindade/gun-violence-twitter-forecasting/data/gun_violence_incidents.csv"
GEOCODE_CACHE_PATH = os.path.join(OUTPUT_DIR, "address_to_latlon_cache.csv")

OUTPUT_FILE = os.path.join(OUTPUT_DIR, "gv_dates_filtered_tweets.csv.gz")
NUM_CORES = min(20, os.cpu_count() or 1)

# Cap incident / tweet years (geotweet archives end in 2023)
MAX_YEAR = 2023

# Spatial filter: keep tweets within this radius of the incident
RADIUS_MILES = 4.0
EARTH_RADIUS_MILES = 3958.7613

# 0 = incident day, -1 = one day before, -2 = two days before
DAY_OFFSETS = (0, -1, -2)

# Filled by later cells (needed as globals for Pool workers on Linux fork)
ALLOWED_DAYS = set()
DAY_TO_INCIDENTS = {}
TARGET_YEARS = []
INCIDENTS_DF = None

try:
    os.makedirs(OUTPUT_DIR, exist_ok=True)
except OSError as e:
    print(f"Note: could not create OUTPUT_DIR yet ({e})")

print("Input Directory :", RAW_DATA_DIR)
print("Output Directory:", OUTPUT_DIR)
print("latLong file    :", LATLONG_PATH)
print("Geocode cache   :", GEOCODE_CACHE_PATH)
print("Output file     :", OUTPUT_FILE)
print("Workers         :", NUM_CORES)
print("Window          : incident day + 1 day before + 2 days before")

# --- Local fallback (run from repo root or notebooks/) ---
from pathlib import Path as _Path
_LOCAL = _Path.cwd().resolve()
if _LOCAL.name == "notebooks":
    _LOCAL = _LOCAL.parent
_local_latlong = _LOCAL / "data" / "input" / "gun_violence_incidents.csv"
_local_geocode = _LOCAL / "data" / "input" / "address_to_latlon_cache.csv"
if _local_latlong.exists():
    LATLONG_PATH = str(_local_latlong)
if _local_geocode.exists():
    GEOCODE_CACHE_PATH = str(_local_geocode)

print("Radius          :", RADIUS_MILES, "miles")
print("Max year        :", MAX_YEAR)
print("RAW exists      :", os.path.isdir(RAW_DATA_DIR))


## 1. Load `gun_violence_incidents.csv` (skip missing Address) & geocode


In [ ]:
def resolve_data_path(path: str, local_name: str) -> str:
    if os.path.exists(path):
        return path
    local = os.path.join(os.getcwd(), local_name)
    if os.path.exists(local):
        print(f"Cluster path missing; using local file: {local}")
        return local
    raise FileNotFoundError(f"{local_name} not found at:\n  {path}\n  or {local}")


def resolve_cache_path(path: str) -> str:
    """Prefer existing cache (cluster or local); else writable OUTPUT_DIR; else cwd."""
    local = os.path.join(os.getcwd(), os.path.basename(path))
    if os.path.exists(path):
        return path
    if os.path.exists(local):
        print(f"Using local geocode cache path: {local}")
        return local
    parent = os.path.dirname(path)
    if parent and os.path.isdir(parent) and os.access(parent, os.W_OK):
        return path
    print(f"Using local geocode cache path: {local}")
    return local


def load_latlong_incidents(path: str) -> pd.DataFrame:
    path = resolve_data_path(path, "data/input/gun_violence_incidents.csv")
    raw = pd.read_csv(path)
    required = {"Incident Date", "State", "City Or County", "Address"}
    missing_cols = required - set(raw.columns)
    if missing_cols:
        raise ValueError(f"Missing columns in gun_violence_incidents.csv: {sorted(missing_cols)}")

    df = raw.copy()
    df["Incident Date"] = pd.to_datetime(df["Incident Date"], format="mixed", errors="coerce")
    n_bad_date = int(df["Incident Date"].isna().sum())

    # Treat NaN / blank / literal "nan" as missing
    addr_raw = df["Address"]
    addr = addr_raw.astype(str).str.strip()
    missing_addr = addr_raw.isna() | (addr == "") | (addr.str.lower() == "nan")
    n_missing_addr = int(missing_addr.sum())

    df = df.loc[~missing_addr].copy()
    df = df.dropna(subset=["Incident Date"])
    df = df[df["Incident Date"].dt.year <= MAX_YEAR].copy()
    df["Incident Date"] = df["Incident Date"].dt.normalize()
    df["Address"] = df["Address"].astype(str).str.strip()
    df["State"] = df["State"].astype(str).str.strip()
    df["City Or County"] = df["City Or County"].astype(str).str.strip()
    df["query"] = (
        df["Address"] + ", " + df["City Or County"] + ", " + df["State"] + ", USA"
    )
    df = df.reset_index(drop=True)
    df["incident_id"] = df.index.astype(int)

    print(f"Loaded latLong from: {path}")
    print(f"  rows in CSV           : {len(raw)}")
    print(f"  unparseable dates     : {n_bad_date}")
    print(f"  skipped missing addr  : {n_missing_addr}")
    print(f"  kept (year <= {MAX_YEAR}) : {len(df)}")
    print(f"  date range            : {df['Incident Date'].min().date()} → {df['Incident Date'].max().date()}")
    display(df.head(10))
    return df


def load_geocode_cache(cache_path: str) -> dict:
    if not os.path.exists(cache_path):
        return {}
    cache_df = pd.read_csv(cache_path)
    out = {}
    for _, row in cache_df.iterrows():
        q = str(row["query"])
        lat = row.get("lat")
        lon = row.get("lon")
        if pd.isna(lat) or pd.isna(lon):
            out[q] = (None, None)
        else:
            out[q] = (float(lat), float(lon))
    print(f"Loaded geocode cache: {len(out)} queries from {cache_path}")
    return out


def save_geocode_cache(cache: dict, cache_path: str) -> None:
    rows = [{"query": q, "lat": lat, "lon": lon} for q, (lat, lon) in sorted(cache.items())]
    pd.DataFrame(rows).to_csv(cache_path, index=False)
    print(f"Saved geocode cache: {len(rows)} queries → {cache_path}")


def geocode_one_arcgis(query: str, timeout: float = 15.0):
    """Single-address fallback via ArcGIS World Geocoding (no API key)."""
    import urllib.parse
    import urllib.request

    params = urllib.parse.urlencode(
        {
            "f": "json",
            "singleLine": query,
            "maxLocations": 1,
            "outFields": "Match_addr",
        }
    )
    url = (
        "https://geocode.arcgis.com/arcgis/rest/services/World/GeocodeServer/"
        f"findAddressCandidates?{params}"
    )
    req = urllib.request.Request(url, headers={"User-Agent": "gv-twitter-pipeline/1.0"})
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        payload = json.loads(resp.read().decode("utf-8"))
    cands = payload.get("candidates") or []
    if not cands:
        return None, None
    loc = cands[0].get("location") or {}
    lat, lon = loc.get("y"), loc.get("x")
    if lat is None or lon is None:
        return None, None
    return float(lat), float(lon)


def geocode_batch_census(records: list[dict], chunk_size: int = 9000) -> dict:
    """
    Batch-geocode US addresses with the Census Bureau addressbatch API.

    records: list of {"query", "street", "city", "state"}
    Returns {query: (lat, lon) | (None, None)}.
    """
    import csv as _csv
    import io
    import urllib.request
    import uuid

    url = "https://geocoding.geo.census.gov/geocoder/locations/addressbatch"
    results = {}

    for start in range(0, len(records), chunk_size):
        chunk = records[start : start + chunk_size]
        id_to_query = {str(i): rec["query"] for i, rec in enumerate(chunk)}

        buf = io.StringIO()
        writer = _csv.writer(buf, quoting=_csv.QUOTE_ALL)
        for i, rec in enumerate(chunk):
            writer.writerow([str(i), rec["street"], rec["city"], rec["state"], ""])
        payload = buf.getvalue().encode("utf-8")

        boundary = f"----GeoBoundary{uuid.uuid4().hex}"
        body = (
            f"--{boundary}\r\n"
            f'Content-Disposition: form-data; name="benchmark"\r\n\r\n'
            f"Public_AR_Current\r\n"
            f"--{boundary}\r\n"
            f'Content-Disposition: form-data; name="vintage"\r\n\r\n'
            f"Current_Current\r\n"
            f"--{boundary}\r\n"
            f'Content-Disposition: form-data; name="addressFile"; filename="addresses.csv"\r\n'
            f"Content-Type: text/csv\r\n\r\n"
        ).encode("utf-8") + payload + f"\r\n--{boundary}--\r\n".encode("utf-8")

        req = urllib.request.Request(
            url,
            data=body,
            headers={
                "Content-Type": f"multipart/form-data; boundary={boundary}",
                "User-Agent": "gv-twitter-pipeline/1.0",
            },
            method="POST",
        )
        print(f"  Census batch chunk {start // chunk_size + 1}: {len(chunk)} addresses...")
        with urllib.request.urlopen(req, timeout=600) as resp:
            text = resp.read().decode("utf-8", errors="replace")

        for line in text.splitlines():
            if not line.strip():
                continue
            try:
                row = next(_csv.reader([line]))
            except Exception:
                continue
            if len(row) < 3:
                continue
            rid = row[0].strip()
            q = id_to_query.get(rid)
            if q is None:
                continue

            # Census columns: id, input, status, ... , "lon,lat", ...
            status = row[2].strip().lower() if len(row) > 2 else ""
            lat = lon = None
            if status in {"match", "tie"}:
                for cell in row[3:]:
                    if "," in cell:
                        maybe = cell.split(",")
                        if len(maybe) == 2:
                            try:
                                lon_f = float(maybe[0])
                                lat_f = float(maybe[1])
                                if -180 <= lon_f <= 180 and -90 <= lat_f <= 90:
                                    lon, lat = lon_f, lat_f
                                    break
                            except ValueError:
                                pass
            results[q] = (lat, lon)

        for rec in chunk:
            results.setdefault(rec["query"], (None, None))

    return results


def geocode_incidents(df: pd.DataFrame, cache_path: str) -> pd.DataFrame:
    """Geocode unique address queries. Prefer Census batch; ArcGIS for leftovers. Cache to disk."""
    cache_path = resolve_cache_path(cache_path)
    cache = load_geocode_cache(cache_path)

    # Unique queries with structured street/city/state for Census batch
    uniq = (
        df[["query", "Address", "City Or County", "State"]]
        .drop_duplicates(subset=["query"])
        .rename(columns={"Address": "street", "City Or County": "city", "State": "state"})
    )
    unique_queries = uniq["query"].tolist()
    todo_mask = ~uniq["query"].isin(cache.keys())
    todo_df = uniq.loc[todo_mask]
    todo_records = todo_df.to_dict(orient="records")

    print(f"Unique address queries : {len(unique_queries)}")
    print(f"Already cached         : {len(unique_queries) - len(todo_records)}")
    print(f"Need geocoding         : {len(todo_records)}")

    if todo_records:
        try:
            batch_hits = geocode_batch_census(todo_records)
            for q, coords in batch_hits.items():
                cache[q] = coords
            save_geocode_cache(cache, cache_path)
            still = [r for r in todo_records if cache.get(r["query"], (None, None))[0] is None]
            print(f"Census matched         : {len(todo_records) - len(still)} / {len(todo_records)}")
        except Exception as e:
            print(f"Census batch failed ({e}); falling back to ArcGIS one-by-one.")
            still = todo_records

        if still:
            print(f"ArcGIS fallback for    : {len(still)} addresses")
            for i, rec in enumerate(tqdm(still, desc="ArcGIS geocode"), start=1):
                q = rec["query"]
                try:
                    cache[q] = geocode_one_arcgis(q)
                except Exception:
                    cache[q] = (None, None)
                time.sleep(0.15)
                if i % 100 == 0 or i == len(still):
                    save_geocode_cache(cache, cache_path)
    else:
        print("All queries already in cache — skipping geocoder calls.")

    coords = [cache.get(q, (None, None)) for q in df["query"]]
    out = df.copy()
    out["incident_lat"] = [c[0] for c in coords]
    out["incident_lon"] = [c[1] for c in coords]
    n_ok = int(out["incident_lat"].notna().sum())
    n_fail = len(out) - n_ok
    out = out.dropna(subset=["incident_lat", "incident_lon"]).reset_index(drop=True)

    print(f"Incidents with coordinates : {n_ok}")
    print(f"Dropped (geocode failed)   : {n_fail}")
    print(f"Final geocoded incidents   : {len(out)}")
    display(out.head(10))
    return out


INCIDENTS_DF = load_latlong_incidents(LATLONG_PATH)
INCIDENTS_DF = geocode_incidents(INCIDENTS_DF, GEOCODE_CACHE_PATH)


## 2. Build allowed tweet days (t, t−1, t−2) with incident coords


In [ ]:
def build_day_index(incidents: pd.DataFrame):
    """Map each calendar day → list of geocoded incidents whose window covers that day."""
    allowed = set()
    day_to_incidents = {}

    for _, row in incidents.iterrows():
        t = pd.Timestamp(row["Incident Date"]).normalize()
        incident = {
            "incident_ts": t,
            "lat": float(row["incident_lat"]),
            "lon": float(row["incident_lon"]),
            "address": row["Address"],
            "city": row["City Or County"],
            "state": row["State"],
        }
        for offset in DAY_OFFSETS:
            d = (t + pd.Timedelta(days=offset)).date()
            allowed.add(d)
            day_to_incidents.setdefault(d, []).append(incident)

    allowed = {d for d in allowed if d.year <= MAX_YEAR}
    day_to_incidents = {d: v for d, v in day_to_incidents.items() if d in allowed}
    years = sorted({str(d.year) for d in allowed})

    print(f"Geocoded incidents          : {len(incidents)}")
    print(f"Allowed tweet calendar days : {len(allowed)}")
    print(f"Target years to scan        : {years}")
    return allowed, day_to_incidents, years


ALLOWED_DAYS, DAY_TO_INCIDENTS, TARGET_YEARS = build_day_index(INCIDENTS_DF)


## 3. File helpers (date prefilter + 3-mile spatial filter)


In [ ]:
def haversine_miles(lat1, lon1, lat2, lon2):
    """Vectorized great-circle distance in miles. lat/lon in degrees."""
    lat1 = np.radians(np.asarray(lat1, dtype=float))
    lon1 = np.radians(np.asarray(lon1, dtype=float))
    lat2 = np.radians(np.asarray(lat2, dtype=float))
    lon2 = np.radians(np.asarray(lon2, dtype=float))
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    return 2.0 * EARTH_RADIUS_MILES * np.arcsin(np.sqrt(np.clip(a, 0.0, 1.0)))


def should_process_file(file_path: str) -> bool:
    """Skip archives whose filename date is outside allowed days (chicago.ipynb idea)."""
    basename = os.path.basename(file_path)
    match = re.search(r"(\d{4})_(\d{2})_(\d{2})", basename)
    if not match:
        return True
    file_day = pd.Timestamp(
        year=int(match.group(1)),
        month=int(match.group(2)),
        day=int(match.group(3)),
    ).date()
    return file_day in ALLOWED_DAYS


def process_file(file_path: str) -> pd.DataFrame:
    """
    Read one CGA geotweet *.csv.gz and keep rows on allowed window days
    that fall within RADIUS_MILES of a matching incident.
    """
    try:
        if not should_process_file(file_path):
            return pd.DataFrame()

        with gzip.open(file_path, "rt", encoding="utf-8") as f:
            df = pd.read_csv(f, sep="\t", on_bad_lines="skip", dtype=str)

        if "tweet_date" not in df.columns and "date" in df.columns:
            df["tweet_date"] = df["date"]
        elif "tweet_date" in df.columns and "date" in df.columns:
            df["tweet_date"] = df["tweet_date"].fillna(df["date"])

        if "text" not in df.columns and "tweet_text" in df.columns:
            df["text"] = df["tweet_text"]

        if "tweet_date" not in df.columns:
            return pd.DataFrame()
        if "latitude" not in df.columns or "longitude" not in df.columns:
            return pd.DataFrame()

        df["tweet_date"] = pd.to_datetime(df["tweet_date"], errors="coerce")
        df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
        df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")
        df = df.dropna(subset=["tweet_date", "latitude", "longitude"])
        if "text" in df.columns:
            df = df.dropna(subset=["text"])

        df["just_date"] = df["tweet_date"].dt.date
        df = df[df["just_date"].isin(ALLOWED_DAYS)].copy()
        if df.empty:
            return pd.DataFrame()

        matched = []
        for just_date, group in df.groupby("just_date", sort=False):
            for incident in DAY_TO_INCIDENTS.get(just_date, []):
                piece = group.copy()
                dist = haversine_miles(
                    piece["latitude"].to_numpy(),
                    piece["longitude"].to_numpy(),
                    incident["lat"],
                    incident["lon"],
                )
                keep = dist <= RADIUS_MILES
                if not np.any(keep):
                    continue
                piece = piece.iloc[np.flatnonzero(keep)].copy()
                incident_ts = incident["incident_ts"]
                delta = (just_date - incident_ts.date()).days  # 0, -1, or -2
                if delta == 0:
                    piece["tag"] = "t"
                elif delta == -1:
                    piece["tag"] = "t-1"
                elif delta == -2:
                    piece["tag"] = "t-2"
                else:
                    piece["tag"] = f"t{delta}"
                matched.append(piece)

        if not matched:
            return pd.DataFrame()

        out = pd.concat(matched, ignore_index=True)
        out["tweet_date"] = out["tweet_date"].dt.strftime("%Y-%m-%d %H:%M:%S")
        out = out.drop(columns=["just_date"], errors="ignore")
        return out
    except Exception as e:
        # Print in-worker so errors surface under multiprocessing
        print(f"process_file error: {file_path}: {type(e).__name__}: {e}", flush=True)
        return pd.DataFrame()


print("Helpers ready.")
print(f"Spatial radius = {RADIUS_MILES} miles")


## 4. Scan archives & filter (parallel)


In [ ]:
def collect_files(years: list[str]) -> list[str]:
    all_files = []
    print(f"Scanning base source path: {RAW_DATA_DIR}")
    for year in years:
        year_path = os.path.join(RAW_DATA_DIR, year, "**", "*.csv.gz")
        found = glob.glob(year_path, recursive=True)
        all_files.extend(found)
        print(f"  Year {year}: {len(found)} archives")
    print(f"Cumulative archives found: {len(all_files)}")
    return all_files


if not os.path.isdir(RAW_DATA_DIR):
    print("RAW_DATA_DIR not available on this machine.")
    print("This cell is meant to run on Harvard FAS where the CGA geotweet store is mounted.")
    print(f"Missing path: {RAW_DATA_DIR}")
    all_files = []
else:
    all_files = collect_files(TARGET_YEARS)

if not all_files:
    print("No tweet archives to process — skipping parallel filter.")
    results = []
else:
    print(f"\nUsing {NUM_CORES} workers...")
    results = []
    with Pool(processes=NUM_CORES) as pool:
        for chunk in tqdm(
            pool.imap_unordered(process_file, all_files),
            total=len(all_files),
            desc="Filtering geotweets",
            mininterval=5,
        ):
            if chunk is not None and not chunk.empty:
                results.append(chunk)
    print(f"Non-empty chunks: {len(results)}")


## 5. Save filtered tweets


In [ ]:
if not results:
    print("No tweets matched — nothing written.")
else:
    final_df = pd.concat(results, ignore_index=True)
    try:
        os.makedirs(OUTPUT_DIR, exist_ok=True)
    except OSError as e:
        raise RuntimeError(f"Cannot write to OUTPUT_DIR={OUTPUT_DIR}: {e}") from e
    final_df.to_csv(OUTPUT_FILE, index=False, compression="gzip", sep="\t")

    print(f"SUCCESS: {len(final_df)} rows written")
    print(f"Output : {OUTPUT_FILE}")
    if "tag" in final_df.columns:
        print("\nRows by tag (t / t-1 / t-2):")
        display(final_df["tag"].value_counts())
    display(final_df.head(5))
